# PhysioCell placental-villous tutorial

This notebook walks through the nidus-PhysioCell integration end-to-end:

1. Generate `nidus-parameters.xml` from the bundled dataset.
2. Inspect which nidus parameters the example consumes.
3. Reproduce the simulation's glucose-transport kinetics in pure Python
   so the C++/PhysioCell side stays auditable.
4. Run an **IUGR sensitivity scenario**: shrink the placental surface
   area 20% and watch the downstream effect on net glucose flux.

The notebook executes in a couple of seconds and does **not** require
a PhysioCell install — it uses the same NumPy reference kernels the
SBML/CellML round-trip tests use, so any drift between the exported
model and the simulator's kinetics shows up immediately.

## 1. Regenerate `nidus-parameters.xml`

The PhysioCell project's `PhysiCell_settings.xml` `<include>`s
`nidus-parameters.xml` by convention. Regenerating it after a
dataset update is one CLI call:

```bash
nidus export --format physiocell \
    --output docs/examples/physicell_placental_villous/
```

Same effect from Python:

In [ ]:
import nidus
from nidus import export

ds = nidus.load()
params_xml = export.build_physiocell_params(ds)
for line in params_xml.splitlines()[:8]:
    print(line)
print('...')
print(f'total length: {len(params_xml):,} chars, {params_xml.count("<param ")} <param/> entries')

Every nidus parameter shows up as a `<param>` element whose
`description` attribute carries the citation key + tier. That string
is what an auditor (or future-you) greps for when a value is
challenged.

## 2. Which nidus parameters does the example consume?

The villous slice example reads four glucose-transport values
directly:

In [ ]:
consumed_ids = [
    'placental_glucose.glucose_glut1_km_mmol_per_l',
    'placental_glucose.glucose_glut1_vmax_per_area_mmol_per_min_per_m2',
    'placental_glucose.glucose_glut3_km_mmol_per_l',
    'placental_glucose.glucose_glut3_vmax_per_area_mmol_per_min_per_m2',
]
for pid in consumed_ids:
    p = ds[pid]
    print(f'{pid}\n  central={p.value.central} {p.value.units}'
          f'  tier={p.tier}  cite={p.primary_citation.key}')

## 3. Reproduce the GLUT1/GLUT3 kinetics in NumPy

`nidus.export.reference` exposes pure-NumPy implementations of every
submodel. These are the same kernels the SBML/CellML round-trip tests
compare against — so if the C++ side of the example drifts away from
the exported model, this notebook fires.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nidus.export.reference import michaelis_menten_flux

glut1_km = ds['placental_glucose.glucose_glut1_km_mmol_per_l'].value.central
glut1_vmax = ds['placental_glucose.glucose_glut1_vmax_per_area_mmol_per_min_per_m2'].value.central
glut3_km = ds['placental_glucose.glucose_glut3_km_mmol_per_l'].value.central
glut3_vmax = ds['placental_glucose.glucose_glut3_vmax_per_area_mmol_per_min_per_m2'].value.central

substrate = np.linspace(0.1, 20.0, 80)
v_glut1 = michaelis_menten_flux(substrate, km_mmol_per_l=glut1_km, vmax_per_area=glut1_vmax)
v_glut3 = michaelis_menten_flux(substrate, km_mmol_per_l=glut3_km, vmax_per_area=glut3_vmax)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(substrate, v_glut1, label=f'GLUT1 (Km={glut1_km}, Vmax={glut1_vmax})')
ax.plot(substrate, v_glut3, label=f'GLUT3 (Km={glut3_km}, Vmax={glut3_vmax})')
ax.set_xlabel('Glucose [mmol/L]')
ax.set_ylabel('V [mmol/min/m^2]')
ax.set_title('Michaelis-Menten kinetics from the nidus dataset')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## 4. IUGR scenario: 20% smaller placental surface area

A common what-if is: *how much glucose can the placenta still
deliver if the villous surface area runs 20% below the term central
estimate?* In PhysioCell terms, the BioFVM domain has the same
geometry but the integrated surface area in the simulation is
smaller; in the reference kernel, we just multiply the integrated
flux by 0.8.

In [ ]:
term_area = ds['placental_structure.term_area_m2'].value.central  # m^2
maternal_glucose = 5.0  # mmol/L (typical)

v_basal = michaelis_menten_flux(
    np.array([maternal_glucose]),
    km_mmol_per_l=glut1_km, vmax_per_area=glut1_vmax,
)[0]
v_microvillous = michaelis_menten_flux(
    np.array([maternal_glucose]),
    km_mmol_per_l=glut3_km, vmax_per_area=glut3_vmax,
)[0]
v_per_m2 = v_basal + v_microvillous

for scenario, scale in [('normal term', 1.0), ('IUGR (-20% surface area)', 0.8)]:
    total_flux = v_per_m2 * term_area * scale
    print(f'{scenario:30s}  flux ~ {total_flux:5.2f} mmol/min  (area={term_area*scale:.2f} m^2)')

## 5. What to do next

- Read [`README.md`](README.md) in this directory for the build/run
  instructions on the actual PhysioCell side.
- For a wider sensitivity sweep, see `nidus.export.sweep` /
  `nidus.export.write_sweep_csv` in the [API reference](../../api.md).
- For the formal interop contract (annotations, validation,
  submission), see the [spec](https://github.com/clay-good/nidus/blob/main/docs/specs/v0.4/01-mechanistic-modeling-interop.md).